In [1]:
import numpy as np
import torch
from torch import nn, Tensor
import tensorflow as tf
from tensorflow import keras

2025-07-22 19:09:06.141557: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-22 19:09:06.186243: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-22 19:09:06.186293: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-22 19:09:06.186341: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-22 19:09:06.196836: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: A

Below is an implementation of the `HistDense` layer in `pytorch`.

In [2]:
class HistogramLayer(nn.Module):
    def __init__(
        self,
        num_bins: int,
        taumax: float = 100.0,
        trainable: bool = True,
        negative: float = 0.0,
    ) -> None:
        """
        A PyTorch implementation of the custom HistDense layer
        presented in https://doi.org/10.1088/2058-9565/ad3c68
        based on the publicly available GitHub code.
        """
        super().__init__()
        self.taumax = taumax
        self.num_bins = num_bins
        self.trainable = trainable
        self.negative = negative

        width_0 = taumax / num_bins
        shifts_init = torch.linspace(
            -negative * taumax + width_0 / 2, taumax - width_0 / 2, num_bins
        ).reshape(num_bins, 1)

        self.width = nn.Parameter(
            torch.tensor([width_0], dtype=torch.float32), requires_grad=trainable
        )
        self.shifts = nn.Parameter(
            torch.tensor(shifts_init, dtype=torch.float32), requires_grad=trainable
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.histogram_sigmoid(x, self.num_bins, self.shifts, self.width)

    @staticmethod
    def theta(x: Tensor) -> Tensor:
        return (x >= 0).float()

    @staticmethod
    def sigmoid(x: Tensor, factor: float = 20.0) -> Tensor:
        return torch.sigmoid(factor * x)

    @staticmethod
    def theta_prod(x: Tensor, width: float = 1.0) -> Tensor:
        return HistogramLayer.theta(x + width / 2) * HistogramLayer.theta(
            -x + width / 2
        )

    @staticmethod
    def theta_prod_sigmoid(
        x: Tensor, factor: float = 20.0, width: float = 1.0
    ) -> Tensor:
        return HistogramLayer.sigmoid(x + width / 2, factor) * HistogramLayer.sigmoid(
            -x + width / 2, factor
        )

    @staticmethod
    def histogram_sigmoid(
        input_tensor: Tensor,  # shape: (batch_size, dim)
        num_bins: int,
        shifts: Tensor,  # shape: (num_bins, dim)
        width: Tensor,  # scalar, shape (num_bins,), or shape (num_bins, dim)
    ) -> Tensor:
        """
        Differentiable histogram with sigmoid-shaped bins for 2D batched input.

        Returns:
            Tensor of shape (batch_size, num_bins)
        """
        B, D = input_tensor.shape  # batch size and input dim

        # Expand tensors for broadcasting
        input_exp = input_tensor.unsqueeze(1)  # (B, 1, D)
        shifts_exp = shifts.unsqueeze(0)  # (1, num_bins, D)
        input_shifted = input_exp - shifts_exp  # (B, num_bins, D)

        factor = 20.0

        # Handle width broadcasting
        if width.ndim == 0:
            width = width.expand(num_bins, D)  # scalar → (num_bins, D)
        elif width.ndim == 1:
            width = width.unsqueeze(1).expand(num_bins, D)  # (num_bins,) → (num_bins, D)
        elif width.ndim == 2 and width.shape[1] != D:
            raise ValueError(f"Width shape {width.shape} incompatible with dim={D}")

        width = width.unsqueeze(0)  # (1, num_bins, D)

        input_theta = torch.sigmoid(
            (input_shifted + width / 2) * factor
        ) * torch.sigmoid((-input_shifted + width / 2) * factor)  # (B, num_bins, D)

        histogram = input_theta.sum(dim=-1)  # (B, num_bins)

        return histogram

Below is the original implementation of the `HistDense` layer in `tensorflow` taken straight from the GitHub repo.

In [3]:
def theta(x):
  return tf.experimental.numpy.heaviside(x,1)

def sigmoid(x, factor = 20.):
  return tf.nn.sigmoid(factor*x)

def theta_prod(x, width = 1.):
  return theta(x+width/2)*theta(-x+width/2)

def theta_prod_sigmoid(x, factor = 20., width = 1.):
  return sigmoid(x+width/2,factor)*sigmoid(-x+width/2,factor)

# Clear all previously registered custom objects
keras.saving.get_custom_objects().clear()

#@tf.function
@keras.saving.register_keras_serializable(name="histogram_sigmoid")
def histogram_sigmoid(input_tensor, nbins, shifts, width):
  'I made this to have smoother, differentiable functions '
  input_tensor = tf.expand_dims(input_tensor, axis=0)
  units = nbins
  onesies = tf.ones([units,1])

  input_copy = tf.tensordot(onesies, input_tensor, axes = [[1], [0]])
  shifts = tf.tensordot(shifts, tf.math.divide(input_tensor,input_tensor), axes = [[1], [0]])

  input_shifted = input_copy - shifts

  factor = 20
  input_theta= tf.nn.sigmoid((input_shifted+width/2)*factor)*tf.nn.sigmoid((-input_shifted+width/2)*factor)

  #input_theta=theta_prod_sigmoid(input_shifted, width = width)
  histogram = tf.transpose(tf.math.reduce_sum(input_theta,axis=-1))
  return histogram


  ### SIGMOID HISTOGRAM LAYER ###
# Using sigmoids I have well-defined gradients.

@keras.saving.register_keras_serializable()
class MyHistogramLayer_Sigmoid(keras.layers.Layer):
    def __init__(self, nbins=399,taumax=100., trainable = True, negative = 0):
        super().__init__()
        self.taumax = taumax
        self.nbins = nbins
        self.width_0 = taumax/nbins
        self.trainable = trainable
        self.negative = negative

    def build(self, batch_input_shape):
        self.width = self.add_weight(name = 'width',
            shape=([1]),
            initializer=tf.constant_initializer([self.width_0]),
            trainable=self.trainable,
        )
        self.shifts = self.add_weight(name = 'shifts',
            shape=([self.nbins, 1]),
            initializer=tf.constant_initializer(
                np.reshape(np.linspace(-self.negative*(self.taumax)+self.width_0/2,self.taumax-self.width_0/2,self.nbins),[self.nbins,1])
                ),
            trainable=self.trainable        )


    def call(self, inputs):
        return histogram_sigmoid(inputs, self.nbins, self.shifts, self.width)

    def get_config(self):
      return {"nbins": self.nbins, "taumax": self.taumax, "trainable":self.trainable, "negative" : self.negative}
    
    
    # Custom Layer
@keras.saving.register_keras_serializable()
class ExpandDimLayer(tf.keras.layers.Layer):
    def __init__(self, axis=-1, **kwargs):
        super(ExpandDimLayer, self).__init__(**kwargs)
        self.axis = axis
    
    def build(self, input_shape):
        super(ExpandDimLayer, self).build(input_shape)
    
    def call(self, inputs):
        return tf.expand_dims(inputs, axis=self.axis)
    
    def get_config(self):
        config = super(ExpandDimLayer, self).get_config()
        config.update({'axis': self.axis})
        return config

Test that the output of the original implementation matches that of the `PyTorch` implementation.

In [5]:
for _ in range(50):
    for nbins in range(50, 800, 50):
        t = torch.rand((50,))
        hist_tf = MyHistogramLayer_Sigmoid(nbins, taumax=torch.max(t).item(), trainable=False)
        hist_torch = HistogramLayer(nbins, taumax=torch.max(t).item(), trainable=False)
        t_tf = hist_tf(tf.convert_to_tensor(t.numpy()))
        t_torch = hist_torch(t.unsqueeze(0))  # add batch dim for input to hist layer

        if torch.allclose(t_torch, torch.tensor(t_tf.numpy())) != True:
            raise ValueError(f"Values are not equal: {t_tf} != {t_torch[0].tolist()}")

/tmp/ipykernel_77592/1949204704.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(shifts_init, dtype=torch.float32), requires_grad=trainable
